# 01 · Can and Square ph data: what the policy sees and does

**Purpose.** Before training on Can and Square (#10, #11), check what the low-dim BC-RNN
actually receives and outputs: the observation keys of the paper config, the layout of the
14-D `object` vector, the meaning and range of the 7-D action, demo lengths, the initial
object poses the policy must generalize over, and the train/valid split. The split and file
checksums are frozen into a manifest that later chapters read.

| | |
|---|---|
| Track | A (upstream datasets; inspection only, no simulation, no training) |
| Kernel | sim venv (`$WORK/franka-il/venvs/sim`, Jupyter kernel "franka sim"); needs `h5py`, `numpy`, `scipy`, `matplotlib` |
| Inputs | `$HPCWORK/franka-il/data/robomimic_v15/{can,square}/ph/low_dim_v15.hdf5` and `expected.sha256` (read-only) |
| Outputs | `results/data_manifest.json`, `docs/figures/can_square_{lengths,actions,init_poses}.png` |
| Runtime | under a minute |
| Success check | every assertion passes; the manifest lists 200 demos per task with train 180 / valid 20, disjoint |

**How to use it.** Cells marked **Predict** come before the answer. Write your guess in the
chat first, then run the next cells.

## 0. Setup

In [ ]:
import hashlib
import json
import os
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
from scipy.spatial.transform import Rotation

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_ROOT = Path(os.environ["HPCWORK"]) / "franka-il/data/robomimic_v15"
TASKS = ["can", "square"]
FIG_DIR = REPO / "docs" / "figures"
MANIFEST = REPO / "results" / "data_manifest.json"

files = {t: h5py.File(DATA_ROOT / t / "ph" / "low_dim_v15.hdf5", "r") for t in TASKS}
# Paper BC-RNN low-dim inputs (configs/can/bc_rnn_low_dim.json, identical for Square).
cfg = json.loads((REPO / "configs/can/bc_rnn_low_dim.json").read_text())
POLICY_OBS = cfg["observation"]["modalities"]["obs"]["low_dim"]
print("policy inputs:", POLICY_OBS)

## 1. File layout and environment metadata

Each file holds `data/demo_<i>` groups (one per human demonstration) and `mask/` lists of
demo names (the splits). `data.attrs["env_args"]` records how the simulator was configured
when the observations were generated; replay and evaluation rebuild the env from it.

In [ ]:
for t, f in files.items():
    ea = json.loads(f["data"].attrs["env_args"])
    ctrl = ea["env_kwargs"]["controller_configs"]["body_parts"]["right"]
    demo = f["data/demo_0"]
    print(f"{t}: env {ea['env_name']} (robosuite {ea['env_version']}), {len(f['data'])} demos, "
          f"{f['data'].attrs['total']} samples, control {ea['env_kwargs']['control_freq']} Hz")
    print(f"   controller {ctrl['type']}, delta={ctrl['control_delta']}, "
          f"output_max {ctrl['output_max']}, gripper {ctrl['gripper']['type']}")
    print("   demo_0 keys:", list(demo), "| states", demo["states"].shape)
    print("   obs:", {k: demo["obs"][k].shape[1] for k in demo["obs"]})

**Predict (1).** The policy sees `object` as one 14-D vector per step. Both tasks have one
target object (the can, the square nut). What are the 14 numbers? Guess the components and
their order, and which coordinate frame each one is expressed in.

## 2. Decoding the 14-D `object` vector

Check guesses against the simulator state, not against documentation. `states` is the
flattened MuJoCo state `[time, qpos, qvel]`; the object's free joint stores its world
position and a `wxyz` quaternion in `qpos`. Addresses from the robosuite scene (checked in
the replay work): `Can_joint0` at qpos 30 (Can scene also holds milk, bread and cereal),
`SquareNut_joint0` at qpos 9.

In [ ]:
QPOS_ADDR = {"can": 30, "square": 9}

for t, f in files.items():
    d = f["data/demo_0"]
    a = 1 + QPOS_ADDR[t]                      # +1: states[0] is time
    pos, quat_wxyz = d["states"][0, a:a + 3], d["states"][0, a + 3:a + 7]
    obj = d["obs/object"][0]
    print(f"{t}: object[0]  = {np.round(obj, 3)}")
    print(f"   qpos pos  = {np.round(pos, 3)}   qpos quat (wxyz) = {np.round(quat_wxyz, 3)}")
    assert np.allclose(obj[7:10], pos, atol=1e-6)
    assert np.allclose(obj[10:14], np.roll(quat_wxyz, -1), atol=1e-6)   # stored as xyzw
print("object[7:10] = world position, object[10:14] = world quaternion (xyzw)")

The first seven numbers are not world coordinates. robosuite names the four parts
`<obj>_to_robot0_eef_pos`, `<obj>_to_robot0_eef_quat`, `<obj>_pos`, `<obj>_quat`, so the
first two should be the object pose **in the gripper frame**: `R_eefᵀ (p_obj − p_eef)`.
Test that on every step of every demo, and also against the *previous* step's object
position (robosuite computes observations from cached sensor values).

In [ ]:
def rel_pos(eef_pos, eef_quat_xyzw, obj_pos):
    # Object position expressed in the gripper frame, for each row.
    R = Rotation.from_quat(eef_quat_xyzw).as_matrix()
    return np.einsum("nji,nj->ni", R, obj_pos - eef_pos)

for t, f in files.items():
    err_same, err_lag, err_t0 = [], [], []
    for d in f["data"].values():
        O, E, Q = d["obs/object"][:], d["obs/robot0_eef_pos"][:], d["obs/robot0_eef_quat"][:]
        same = np.linalg.norm(rel_pos(E, Q, O[:, 7:10]) - O[:, :3], axis=1)
        lag = np.linalg.norm(rel_pos(E[1:], Q[1:], O[:-1, 7:10]) - O[1:, :3], axis=1)
        err_t0.append(same[0])
        err_same.extend(same[1:])
        err_lag.extend(lag)
    q = lambda v: f"median {np.median(v) * 1000:6.1f} mm, p99 {np.percentile(v, 99) * 1000:6.1f} mm"
    print(f"{t}: step 0                          {q(err_t0)}")
    print(f"   steps >=1, object pos at step t   {q(err_same)}")
    print(f"   steps >=1, object pos at step t-1 {q(err_lag)}")

**Reading the result.**

- `object[0:7]` is the object pose relative to the gripper, `object[7:14]` the world pose.
  Both tasks share this layout, so one config serves both.
- The relative part is computed from the **previous step's** object position with the
  current gripper pose (the "t−1" error is ~0 in both tasks). Our hypothesis is robosuite's
  observable caching; it is a measured property of these files, not something we found
  documented. It matters little while the object rests, but by up to centimetres while it is
  carried.
- At **step 0** there is no previous value and the relative part is unrelated to the
  object (~0.9 m error in every demo). The evaluation env produces its own first
  observation the same way, so this is a quirk shared by training and evaluation rather
  than a train/test mismatch, but it is one more reason not to trust the very first action.

## 3. Actions

**Predict (2).** The controller is `OSC_POSE` with `control_delta=True`, input range ±1 and
output range ±0.05 m / ±0.5 rad. What does an action of `[1, 0, 0, 0, 0, 0, -1]` ask the
robot to do in one 20 Hz step? Do you expect human teleop actions to spread evenly over
[−1, 1], or to pile up somewhere?

In [ ]:
DIM_NAMES = ["dx", "dy", "dz", "drx", "dry", "drz", "grip"]
acts = {t: np.concatenate([d["actions"][:] for d in f["data"].values()]) for t, f in files.items()}

for t, A in acts.items():
    print(f"{t}: {len(A)} actions")
    print("   min  ", np.round(A.min(0), 2))
    print("   max  ", np.round(A.max(0), 2))
    print("   |a|=1 share", np.round((np.abs(A) >= 0.999).mean(0), 3))
    print("   gripper values", np.unique(A[:, 6]))

fig, axes = plt.subplots(2, 7, figsize=(15, 4), sharex=True)
for row, t in enumerate(TASKS):
    for j, ax in enumerate(axes[row]):
        ax.hist(acts[t][:, j], bins=60, range=(-1, 1), color=f"C{row}")
        ax.set_yscale("log")
        ax.set_title(f"{t} {DIM_NAMES[j]}", fontsize=9)
fig.suptitle("Action distribution per dimension (log count)")
fig.tight_layout()
fig.savefig(FIG_DIR / "can_square_actions.png", dpi=90)

**Reading the result.** Translation is a normalized delta: 1 means 5 cm in that world axis
per step, which the controller tracks with finite gains, so the hand moves less. The gripper
is binary (−1 open, 1 closed): a GMM policy can represent that as two modes, a plain
regression would average it. Look at how much mass sits at exactly ±1 (clipping during
teleop; for Can one translation axis is saturated in over a third of all steps: which one,
and why?) and how narrow the rotation dims are; that narrowness is part of why low-dim BC
works with little data.

## 4. Demo lengths and the evaluation horizon

**Predict (3).** Lift demos have median 48 steps. Guess the medians for Can and Square. The
evaluation uses horizon 400 (20 s). Is that tight anywhere?

In [ ]:
lengths = {t: np.array([d["actions"].shape[0] for d in f["data"].values()]) for t, f in files.items()}
for t, L in lengths.items():
    print(f"{t}: median {np.median(L):.0f}, p90 {np.percentile(L, 90):.0f}, max {L.max()}, "
          f"min {L.min()} steps -> max uses {L.max() / 400:.0%} of the 400-step horizon")

fig, ax = plt.subplots(figsize=(6, 3))
for i, t in enumerate(TASKS):
    ax.hist(lengths[t], bins=30, alpha=0.6, label=t, color=f"C{i}")
ax.axvline(400, color="k", ls="--", lw=1)
ax.text(395, ax.get_ylim()[1] * 0.9, "eval horizon", ha="right", fontsize=8)
ax.set_xlabel("demo length (steps at 20 Hz)")
ax.set_ylabel("demos")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "can_square_lengths.png", dpi=90)

## 5. Initial object poses

The policy is evaluated on fresh random initial states. The 200 demos show the range it has
seen: object position (x, y) and yaw at step 0.

In [ ]:
def yaw_deg(quat_xyzw):
    return Rotation.from_quat(quat_xyzw).as_euler("xyz", degrees=True)[:, 2]

init = {t: np.stack([d["obs/object"][0, 7:14] for d in f["data"].values()]) for t, f in files.items()}

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, t in zip(axes, TASKS):
    P = init[t]
    sc = ax.scatter(P[:, 0], P[:, 1], c=yaw_deg(P[:, 3:]), cmap="twilight", s=14,
                    vmin=-180, vmax=180)
    ax.set_title(f"{t}: initial object x, y (colour = yaw)")
    ax.set_xlabel("x [m]")
    ax.set_ylabel("y [m]")
    centre = (P[:, :2].max(0) + P[:, :2].min(0)) / 2      # same 40 cm window for both tasks
    ax.set_xlim(centre[0] - 0.2, centre[0] + 0.2)
    ax.set_ylim(centre[1] - 0.2, centre[1] + 0.2)
    ax.set_aspect("equal")
    span = P[:, :2].max(0) - P[:, :2].min(0)
    print(f"{t}: x,y span {np.round(span * 100, 1)} cm, "
          f"yaw range {np.ptp(yaw_deg(P[:, 3:])):.0f} deg")
fig.colorbar(sc, ax=axes, label="yaw [deg]")
fig.savefig(FIG_DIR / "can_square_init_poses.png", dpi=90)

## 6. Splits and the frozen manifest

robomimic trains on `mask/train` and validates on `mask/valid` (`train.hdf5_filter_key`
and `hdf5_validation_filter_key` in the config). The 20 % and 50 % masks are the data-budget
subsets for phase 3. Check that the splits are disjoint, cover all demos, and that the
smaller budgets are nested inside the larger ones; then record names, counts and a hash of
each mask so any later run can prove which demos it used.

In [ ]:
def mask(f, key):
    return sorted(x.decode() for x in f["mask"][key][:])


def sha256(path, chunk=1 << 22):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        while block := fh.read(chunk):
            h.update(block)
    return h.hexdigest()


print("config filter keys:", cfg["train"]["hdf5_filter_key"], cfg["train"]["hdf5_validation_filter_key"])
manifest = {"dataset": "robomimic v0.5 ph low_dim_v15 (HF revision 74fa0184)", "tasks": {}}
for t, f in files.items():
    demos = set(f["data"])
    m = {k: mask(f, k) for k in f["mask"]}
    tr, va = set(m["train"]), set(m["valid"])
    assert not tr & va and tr | va == demos
    for p in ("20_percent", "50_percent"):
        assert set(m[f"{p}_train"]) <= tr and set(m[f"{p}_valid"]) <= va
        assert set(m[f"{p}_train"]) | set(m[f"{p}_valid"]) == set(m[p])
    assert set(m["20_percent"]) <= set(m["50_percent"]), "budgets not nested"

    path = DATA_ROOT / t / "ph" / "low_dim_v15.hdf5"
    expected = dict(line.split()[::-1] for line in (path.parent / "expected.sha256").read_text().splitlines())
    digest = sha256(path)
    assert digest == expected[path.name], "file does not match the recorded SHA256"
    manifest["tasks"][t] = {
        "file": f"robomimic_v15/{t}/ph/{path.name}",
        "sha256": digest,
        "env_name": json.loads(f["data"].attrs["env_args"])["env_name"],
        "n_demos": len(demos),
        "n_samples": int(f["data"].attrs["total"]),
        "length_median_max": [int(np.median(lengths[t])), int(lengths[t].max())],
        "masks": {k: {"n": len(v), "sha1": hashlib.sha1(",".join(v).encode()).hexdigest()[:12]}
                  for k, v in sorted(m.items())},
    }
    print(t, {k: v["n"] for k, v in manifest["tasks"][t]["masks"].items()})

MANIFEST.write_text(json.dumps(manifest, indent=1) + "\n")
print("wrote", MANIFEST.relative_to(REPO))

**Note on the budgets.** The assertions confirm that the 20 % subset lies inside the 50 %
subset, which lies inside the full split. A phase-3 data-budget curve therefore varies
"how much data" without also swapping which demos are used.

## 7. Summary

- Inputs: 3 + 4 + 2 robot dims plus the 14-D object vector = 23 numbers per step, history 10.
- `object` = [pose in gripper frame (one step stale; meaningless at step 0), world pose].
- Actions: normalized OSC deltas in [−1, 1] plus a binary gripper.
- Lengths and init ranges above; the frozen split is in `results/data_manifest.json`.

Next chapter: `02_bc_rnn_baseline` reads the paired evaluation (`results/*_eval_episodes.csv`).